In [1]:
!pip install lightgbm

In [2]:
import pandas as pd
import numpy as np
import pickle

from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("f1_ml_ready_dataset.csv")

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

target_column = "positionOrder"   

df = df.dropna(subset=[target_column])

X = df.drop(columns=[target_column])
y = df[target_column]

X = X.select_dtypes(include=[np.number])

print("\nFeature matrix shape:", X.shape)
print("Target shape:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("\nTraining samples:", X_train.shape)
print("Testing samples:", X_test.shape)

lgbm_model = LGBMRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgbm_model.fit(X_train, y_train)

print("\nLightGBM model training completed.")

y_pred = lgbm_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nModel Performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

feature_importance = pd.Series(
    lgbm_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nTop 10 Important Features:")
print(feature_importance.head(10))

with open("f1_lightgbm_model.pkl", "wb") as f:
    pickle.dump(lgbm_model, f)

print("\nModel saved as f1_lightgbm_model.pkl")

Dataset shape: (26759, 53)
Columns:
['raceId', 'driverId', 'constructorId', 'number_x', 'grid', 'milliseconds', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'number_y', 'driver_nationality', 'constructor_name', 'constructor_nationality', 'year', 'round', 'circuitId', 'race_name', 'url', 'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time', 'quali_date', 'quali_time', 'sprint_date', 'sprint_time', 'driver_age', 'finished_race', 'driver_experience', 'constructor_experience', 'driver_circuit_experience', 'driver_avg_finish_before_race', 'driver_avg_points_before_race', 'driver_points_last_3', 'driver_points_last_5', 'driver_finish_last_3', 'driver_finish_last_5', 'previous_race_finish', 'previous_race_points', 'previous_grid', 'constructor_avg_finish_before_race', 'constructor_avg_points_before_race', 'constructor_points_last_3', 'constructor_points_last_5', 'season_driver_points_before_race', 'season_constructor_points_before_race', 'season_race_number_for_driver', 'sea

In [3]:
import pandas as pd
import numpy as np
import pickle
import warnings

from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

df = pd.read_csv("f1_ml_ready_dataset.csv")

target_column = "positionOrder"

df = df.dropna(subset=[target_column])

X = df.drop(columns=[target_column])
y = df[target_column]

X = X.select_dtypes(include=[np.number])

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)

lgbm_model = LGBMRegressor(
    random_state=42,
    verbose=-1
)

param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "max_depth": [-1, 3, 4, 5, 6, 8, 10],
    "num_leaves": [15, 31, 50, 70, 100],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_samples": [5, 10, 20, 30]
}

random_search = RandomizedSearchCV(
    estimator=lgbm_model,
    param_distributions=param_dist,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

best_lgbm = random_search.best_estimator_

print("\nBest Parameters:")
print(random_search.best_params_)

y_pred = best_lgbm.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\nTuned LightGBM Performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

feature_importance = pd.Series(
    best_lgbm.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nTop 10 Important Features:")
print(feature_importance.head(10))

with open("f1_lightgbm_tuned.pkl", "wb") as f:
    pickle.dump(best_lgbm, f)

print("\nModel saved as f1_lightgbm_tuned.pkl")

Feature matrix shape: (26759, 52)
Target shape: (26759,)
Training samples: (21407, 52)
Testing samples: (5352, 52)
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Best Parameters:
{'subsample': 0.9, 'num_leaves': 50, 'n_estimators': 300, 'min_child_samples': 20, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 0.8}

Tuned LightGBM Performance:
MAE: 3.3732658292514635
RMSE: 4.428905118033242
R²: 0.6777568743948468

Top 10 Important Features:
grid                                  524
url                                   428
rank                                  376
raceId                                283
constructor_experience                277
constructor_avg_finish_before_race    273
driver_avg_finish_before_race         252
year                                  232
driver_finish_last_5                  230
circuitId                             229
dtype: int32

Model saved as f1_lightgbm_tuned.pkl
